# Movie Review Sentiment Analysis using LSTM (IMDb Dataset)

**Pipeline:** data loading → text preprocessing → tokenization → sequence padding
→ embeddings → Bidirectional LSTM → training → evaluation → save artifacts for a
Streamlit app.

Run all cells top to bottom. Training takes a few minutes on CPU (faster on GPU).


## 1. Setup & Imports

In [ ]:
!pip install -q tensorflow scikit-learn streamlit matplotlib

In [ ]:
import json
import pickle
import re
import string

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)

## 2. Configuration

Tweak these to trade off speed vs. accuracy.

In [ ]:
VOCAB_SIZE = 15000       # max words kept in the tokenizer
MAX_LEN = 200             # pad/truncate every review to this many tokens
EMBEDDING_DIM = 128
LSTM_UNITS = 64
BATCH_SIZE = 128
EPOCHS = 8

## 3. Load the IMDb Dataset

Keras ships a pre-indexed version of the IMDb dataset (25k train / 25k test,
balanced positive/negative). We decode the integer sequences back to raw text
so we can fit our **own** tokenizer on real text — this is what lets the
Streamlit app later tokenize arbitrary user-written reviews with the exact
same tokenizer used at training time.

In [ ]:
(x_train_idx, y_train), (x_test_idx, y_test) = imdb.load_data(num_words=None)

word_index = imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word[0] = "<pad>"
index_to_word[1] = "<start>"
index_to_word[2] = "<unk>"
index_to_word[3] = "<unused>"

def decode(seq):
    return " ".join(index_to_word.get(i, "<unk>") for i in seq)

x_train_text = [decode(seq) for seq in x_train_idx]
x_test_text = [decode(seq) for seq in x_test_idx]

print(f"Train reviews: {len(x_train_text)}, Test reviews: {len(x_test_text)}")
print("\nSample raw review:\n", x_train_text[0][:300], "...")
print("\nLabel (0=negative, 1=positive):", y_train[0])

## 4. Text Preprocessing

Lowercase, strip HTML line breaks / URLs / punctuation / digits, normalize
whitespace.

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)          # html line breaks
    text = re.sub(r"http\S+|www\S+", " ", text)      # urls
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)                 # numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

x_train_clean = [clean_text(t) for t in x_train_text]
x_test_clean = [clean_text(t) for t in x_test_text]

print("Before:", x_train_text[0][:150])
print("\nAfter :", x_train_clean[0][:150])

## 5. Tokenization

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(x_train_clean)

x_train_seq = tokenizer.texts_to_sequences(x_train_clean)
x_test_seq = tokenizer.texts_to_sequences(x_test_clean)

print("Example cleaned text:", x_train_clean[0][:100])
print("Example token sequence:", x_train_seq[0][:20])
print("Vocabulary size used:", min(VOCAB_SIZE, len(tokenizer.word_index) + 1))

## 6. Sequence Padding

In [ ]:
x_train_pad = pad_sequences(x_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
x_test_pad = pad_sequences(x_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

y_train = np.array(y_train)
y_test = np.array(y_test)

print("Train shape:", x_train_pad.shape)
print("Test shape :", x_test_pad.shape)
print("\nPadded example:\n", x_train_pad[0])

## 7. Build the Model (Embedding + Bidirectional LSTM)

- **Embedding layer**: learns a dense vector representation for each word.
- **Two stacked Bidirectional LSTM layers**: read the sequence forwards and
  backwards to capture context from both directions.
- **Dense + Dropout**: classification head with regularization.
- **Sigmoid output**: probability of the review being positive.

In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    Bidirectional(LSTM(LSTM_UNITS // 2, dropout=0.2, recurrent_dropout=0.2)),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 8. Train

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history = model.fit(
    x_train_pad,
    y_train,
    validation_split=0.2,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stop],
    verbose=1,
)

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Evaluate on the Test Set

In [ ]:
y_prob = model.predict(x_test_pad, batch_size=BATCH_SIZE).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test ROC-AUC : {auc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["negative", "positive"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["negative", "positive"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.show()

## 10. Try It on Custom Reviews

In [ ]:
def predict_sentiment(text, model=model, tokenizer=tokenizer, max_len=MAX_LEN):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    prob = float(model.predict(padded, verbose=0)[0][0])
    label = "Positive" if prob >= 0.5 else "Negative"
    return label, prob

samples = [
    "This movie was an absolute masterpiece. The acting and score were incredible.",
    "What a waste of time. The plot made no sense and the acting was wooden.",
]
for s in samples:
    label, prob = predict_sentiment(s)
    print(f"[{label} ({prob:.3f})] {s}")

## 11. Save Artifacts for the Streamlit App

Saves `sentiment_lstm.h5` (model), `tokenizer.pkl` (fitted tokenizer), and
`config.json` (run config + metrics) — these are what `app.py` loads.

In [ ]:
model.save("sentiment_lstm.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

config = {
    "vocab_size": VOCAB_SIZE,
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "test_accuracy": float(acc),
    "test_auc": float(auc),
}
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved: sentiment_lstm.h5, tokenizer.pkl, config.json")

## 12. Deploy with Streamlit

This writes `app.py` to disk (the `%%writefile` magic below). Streamlit apps
run as a separate process, not inside a notebook cell — after running this
cell, launch the app from a terminal with:

```bash
streamlit run app.py
```

If you're on Google Colab, use a tunneling tool (e.g. `localtunnel` or
`ngrok`) since Colab can't open a local port directly:

```bash
!npm install -g localtunnel
!streamlit run app.py & npx localtunnel --port 8501
```


In [ ]:
%%writefile app.py
import json
import pickle
import re
import string

import streamlit as st
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

MODEL_PATH = "sentiment_lstm.h5"
TOKENIZER_PATH = "tokenizer.pkl"
CONFIG_PATH = "config.json"


@st.cache_resource(show_spinner="Loading model...")
def load_artifacts():
    model = load_model(MODEL_PATH)
    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)
    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)
    return model, tokenizer, config


def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def predict_sentiment(text, model, tokenizer, max_len):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")
    prob = float(model.predict(padded, verbose=0)[0][0])
    label = "Positive" if prob >= 0.5 else "Negative"
    confidence = prob if prob >= 0.5 else 1 - prob
    return label, confidence, prob


def main():
    st.set_page_config(page_title="Movie Review Sentiment Analyzer", page_icon="\U0001F3AC", layout="centered")
    st.title("\U0001F3AC Movie Review Sentiment Analyzer")
    st.write(
        "An LSTM (Bidirectional) model trained on the IMDb movie reviews "
        "dataset classifies your review as **Positive** or **Negative**."
    )

    try:
        model, tokenizer, config = load_artifacts()
    except (OSError, IOError, FileNotFoundError):
        st.error(
            "Model artifacts not found. Run the training notebook first to "
            "generate sentiment_lstm.h5, tokenizer.pkl, and config.json."
        )
        st.stop()

    with st.sidebar:
        st.header("Model info")
        st.metric("Test accuracy", f"{config['test_accuracy']*100:.2f}%")
        st.metric("Test ROC-AUC", f"{config['test_auc']:.4f}")
        st.caption(f"Vocab size: {config['vocab_size']} | Max sequence length: {config['max_len']}")

    example_reviews = {
        "-- choose an example --": "",
        "Positive example": "This movie was an absolute masterpiece. The acting, "
        "the score, the cinematography -- everything came together beautifully.",
        "Negative example": "What a waste of time. The plot made no sense, the "
        "acting was wooden, and I nearly fell asleep halfway through.",
    }
    choice = st.selectbox("Try an example, or write your own review below:", list(example_reviews.keys()))
    review_text = st.text_area("Your movie review", value=example_reviews[choice], height=180)

    if st.button("Analyze Sentiment", type="primary"):
        if not review_text.strip():
            st.warning("Please enter a review first.")
        else:
            label, confidence, prob = predict_sentiment(review_text, model, tokenizer, config["max_len"])
            if label == "Positive":
                st.success(f"**Prediction: {label}**  (confidence: {confidence*100:.1f}%)")
            else:
                st.error(f"**Prediction: {label}**  (confidence: {confidence*100:.1f}%)")
            st.progress(prob)
            st.caption(f"Raw model output (probability of positive class): {prob:.4f}")


if __name__ == "__main__":
    main()


## 13. (Optional) Launch Streamlit Directly from the Notebook

Only works in a local Jupyter environment (not most hosted notebooks). This
starts the app in the background and prints the local URL to open.

In [ ]:
import subprocess
proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.headless=true"])
print("Streamlit starting... open http://localhost:8501 in your browser.")
print("Run `proc.terminate()` in a new cell to stop the app.")